# V1 decision stack on the labelled week

> **Main question.** Given a raw transaction and a frozen, training-built candidate library, can we pick a small
> number of pack sends with a meaningfully higher probability of capturing an out-of-sample arb?

```text
raw tx -> relevance gate (relevance-v1) -> framed-tx-v2 -> txflat-v2 -> arb classifier
       -> candidate-library lookup (depth k) -> pack ranking -> send top 1-3
```

**Data.** Every labelled partition of the week (`weekly/fetch.py` output: one `ml-prep` directory per partition
plus the frozen library). Partitions are prepared separately; rows are re-keyed to the library's global id
`partition_index << 40 | master_tx_id`, so splits, joins and the library's train window share one id space.

**Protocol.** Slots are split 70 / 15 / 15 in time order: exactly the library's own windows (`windows.json`).
- **Train:** the candidate library (`candidate-packs build-library`), the arb classifier and the pack ranker are all built
  from train only.
- **Validation:** chooses thresholds, send budgets, depth k and live top-L. The library stays frozen.
- **Test:** evaluated once.

The notebook stops if the library's training window extends past the train split.

**Sections.**
1. Data and protocol checks
2. Classifier quality
3. Library availability against depth k
4. Route-recurrence ceiling
5. Static vs ML pack ranking
6. Joint score vs two-stage decisions, at matched send budgets
7. Failure decomposition and conditional results
8. Artifacts

In [ ]:
import json, os
from pathlib import Path

import numpy as np
import polars as pl
from IPython.display import Markdown, display

from firm_ml import data, encoding, metrics, models, plots, policy, splits, week

WEEK = Path(os.environ.get("FIRM_ML_WEEK", "/data/bsc/firm-ml/week/run-001"))   # weekly/fetch.py output
EXPERIMENT = os.environ.get("FIRM_ML_EXPERIMENT", "v1-week-20260925-run-001")
RUN_DIR = Path(os.environ.get("FIRM_ML_RUNS", "../runs")).resolve() / EXPERIMENT
PLOTS = RUN_DIR / "plots"; PLOTS.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("FIRM_ML_THREADS") or os.cpu_count() or 8)
# Optional cap on POSITIVE transactions that train the pack ranker (deterministic in the global id); unset = all.
RANKER_MAX_TXS = int(os.environ["FIRM_ML_RANKER_MAX_TXS"]) if os.environ.get("FIRM_ML_RANKER_MAX_TXS") else None

SPLIT_FRACTIONS = (0.70, 0.15, 0.15)
DEPTHS = [3, 5, 8, 16, 32, 64]          # library depth k
LIVE_TOP = [1, 2, 3]                    # packs sent per decision
BUDGETS = [250, 1000, 4000, 16000]      # sends per chain hour; thresholds are fitted to these on validation
GATE = "relevant"
ARB_PARAMS = {**models.ARB_PARAMS, "num_threads": THREADS}
RANKER_PARAMS = {**models.RANKER_PARAMS, "num_threads": THREADS}

plots.style()
pl.Config.set_tbl_rows(60); pl.Config.set_tbl_cols(30); pl.Config.set_fmt_str_lengths(50); pl.Config.set_tbl_width_chars(240)
W = week.open_week(WEEK)
prep = W.prep                           # feature manifest, pack catalog and library provenance are identical across parts
M = prep.manifest
PPM = M["config"]["unknown_pair_ppm"]
FULL = len(W.parts) == len(W.windows["partitions"])
RESULTS = {"experiment": EXPERIMENT, "week": str(WEEK), "partitions": len(W.parts), "full_week": FULL}
print(WEEK, f"{len(W.parts)}/{len(W.windows['partitions'])} partitions", M["schemas"]["pack_catalog"], M["packs"]["count"], "packs ->", RUN_DIR)
if not FULL:
    display(Markdown("**Partial week loaded: protocol checks that need every partition are skipped.**"))

## 1. Data and protocol checks

In [ ]:
tx = week.load_transactions(W)
packs = data.load_packs(prep)

bounds = W.bounds                       # the library's global windows
if FULL:   # recomputed over every loaded slot, the split must reproduce the windows the library was built on
    again = splits.slot_boundaries(tx["slot"], SPLIT_FRACTIONS)
    assert all(again[s] == bounds[s] for s in splits.SPLITS), (again, bounds)
tx = splits.assign_split(tx, bounds)
splits.check_disjoint(tx)
ids = {s: (int(g["master_tx_id"].min()), int(g["master_tx_id"].max())) for (s,), g in tx.group_by("split")}

# Frozen-library protocol: the catalog (and every train_meta feature) comes from train global ids only.
pw = M["packs"]
assert pw["train_master_tx_id_max"] == W.windows["train"]["global_max"], "library train window differs from the run's"
for s in ("val", "test"):
    if s in ids:
        assert pw["train_master_tx_id_max"] < ids[s][0], f"library window reaches into {s}"
if FULL:
    assert ids["train"][0] <= pw["train_master_tx_id_min"] and pw["train_master_tx_id_max"] <= ids["train"][1]

split_table = tx.group_by("split").agg(
    pl.len().alias("master_txs"), pl.col(GATE).sum().alias("relevant"),
    pl.col("arb_target").is_not_null().sum().alias("labelled"), (pl.col("arb_target") == 1).sum().alias("positives"),
    ((pl.col("arb_target") == 1) & (pl.col(GATE) == 1)).sum().alias("relevant_positives"),
    ((pl.col("block_time").max() - pl.col("block_time").min()) / 3600).alias("chain_hours"),
).sort("split")
HOURS = dict(zip(split_table["split"], split_table["chain_hours"]))
S = week.summary(W)
overview = {
    "partitions": S["partitions"], "master_txs": tx.height,
    "labels": dict(tx.group_by("arb_label").len().iter_rows()),
    "catalog": {k: pw[k] for k in ("catalog_schema", "count", "train_master_tx_id_min", "train_master_tx_id_max", "max_rank")},
    "library_config_sha256": (pw.get("library") or {}).get("config_sha256"),
    "master_id_windows": ids, "pairs": S["pairs"], "relevance_gate_pass_rate": float(tx[GATE].mean()),
    "raw_source": M["raw_source"]["kind"], "v1_format_txs": S["v1_format_txs"],
}
encoder = tx.group_by("arb_label").agg(pl.len().alias("txs"), (pl.col("encode_status") != 0).sum().alias("encoder_cannot_parse"),
                                       (pl.col("encode_status") != 0).mean().alias("share")).sort("arb_label")
RESULTS["encoder_coverage"] = encoder.to_dicts()
display(encoder)
RESULTS.update(overview=overview, splits={"boundaries": bounds, "table": split_table.to_dicts()})
display(split_table)
overview

In [ ]:
# Leakage checks: features come only from the manifest; ids, labels, meta and gate columns are refused;
# pack features are static structure or train-window library metadata (allowed because the window precedes val/test).
data.check_feature_columns(prep, prep.tx_features + prep.pack_features)
for bad in ["slot", "master_tx_id", "arb_target", "pack_target", "pack_rank", "builder_rank", "min_cover_rank_live",
            "n_profitable_routes_seen_in_train", "trigger_keys", GATE]:
    try:
        data.check_feature_columns(prep, [bad]); raise AssertionError(f"{bad} accepted as a feature")
    except ValueError:
        pass
groups = {f["group"] for f in prep.feature_manifest["pack_features"]}
assert groups <= {"static", "train_meta"}, groups
TRAIN_META = prep.pack_features_in("train_meta")
STATIC_PACK = prep.pack_features_in("static")
print(len(prep.tx_features), "tx features;", len(STATIC_PACK), "static pack features;", len(TRAIN_META), "train_meta:", TRAIN_META)

## 2. Classifier quality

LightGBM binary on transaction features only. POSITIVE = 1, CERTIFIED_NEGATIVE = 0, and UNKNOWN is excluded from the
loss. There is no reweighting. Early stopping uses validation PR-AUC. Quality is reported on relevant labelled
transactions, since that is the population the deployed gate passes.

In [ ]:
lab = data.arb_training_rows(tx)
arb_enc = encoding.FeatureEncoder(prep.tx_features, prep.kinds).fit(lab.filter(pl.col("split") == "train"))
part = {s: lab.filter(pl.col("split") == s) for s in splits.SPLITS}
X = {s: arb_enc.transform(part[s]) for s in splits.SPLITS}
Y = {s: part[s]["arb_target"].to_numpy() for s in splits.SPLITS}
G = {s: part[s][GATE].to_numpy() == 1 for s in splits.SPLITS}
arb = models.train_binary(X["train"], Y["train"], X["val"], Y["val"], arb_enc, ARB_PARAMS)
P = {s: arb.predict(X[s], num_iteration=arb.best_iteration) for s in splits.SPLITS}
tx = tx.with_columns(pl.Series("arb_score", np.concatenate(   # chunked: the encoded week does not need to be in memory at once
    [arb.predict(arb_enc.transform(c), num_iteration=arb.best_iteration) for c in tx.iter_slices(2_000_000)])))

rows = []
for s in splits.SPLITS:
    for pop, m in (("all labelled", np.ones_like(G[s])), ("relevant labelled", G[s])):
        r = metrics.classifier_summary(Y[s][m], P[s][m]); r.update(split=s, population=pop); rows.append(r)
arb_summary = pl.DataFrame(rows).select("split", "population", "n", "positives", "base_rate", "pr_auc", "roc_auc")
rng = np.random.default_rng(7)
ctrl = models.train_binary(X["train"], rng.permutation(Y["train"]), X["val"], Y["val"], arb_enc, ARB_PARAMS, rounds=300, early_stopping=50)
RESULTS["arb"] = {"best_iteration": arb.best_iteration, "summary": arb_summary.to_dicts(),
                  "shuffled_label_control_val": metrics.classifier_summary(Y["val"], ctrl.predict(X["val"], num_iteration=ctrl.best_iteration)),
                  "importance": models.importance(arb)[:30]}
plots.pr_curves({"val (relevant)": (Y["val"][G["val"]], P["val"][G["val"]]),
                 "test (relevant, OOS)": (Y["test"][G["test"]], P["test"][G["test"]])},
                "Arb classifier precision-recall (dotted = base rate)", PLOTS / "arb_pr_curve.png")
display(arb_summary)
print("shuffled-label control, val:", RESULTS["arb"]["shuffled_label_control_val"])

## 3. Library OOS availability against depth k

**Availability** at depth k is the share of positive transactions for which some eligible pack among the first k of
its key covers one of their profitable RouteIds.
- **Live:** TOP_LEVEL keys only.
- **Oracle:** all layers.

The oracle column must equal the library repo's own `sweep-library` result (`positive_tx_coverage`) on the same
master-id windows. This cross-checks the RouteId coverage semantics across the two repos.

In [ ]:
rows = []
for s in ("val", "test"):
    t = tx.filter(pl.col("split") == s)
    for mode in ("live", "oracle"):
        for r in policy.availability(t, DEPTHS + [1, 2], mode).iter_rows(named=True):
            rows.append({"split": s, "catalog": "library", "eligibility": mode, **r})
avail = pl.DataFrame(rows).sort("split", "catalog", "eligibility", "k")
RESULTS["availability"] = avail.to_dicts()

# Cross-check against the library repo's sweep (embedded as provenance by ml-prep).
checks = []
for name, entry in M["provenance"].items():
    if not name.startswith("sweep-"):
        continue
    catalog, split = name.removeprefix("sweep-").removesuffix(".json").split("-")
    for row in entry["content"]["sweep"]:
        mine = avail.filter((pl.col("split") == split) & (pl.col("catalog") == catalog) & (pl.col("eligibility") == "oracle") & (pl.col("k") == row["k"]))
        if mine.height:
            checks.append({"sweep": name, "k": row["k"], "library_repo": row["positive_tx_coverage"], "ml_prep": mine["availability"][0],
                           "txs_library_repo": row["positive_txs_covered"], "txs_ml_prep": mine["available"][0]})
checks = pl.DataFrame(checks)
RESULTS["sweep_cross_check"] = checks.to_dicts()
display(checks)
if FULL:   # the sweep covers every partition, so it only matches a full load
    assert checks.height and (checks["txs_library_repo"] == checks["txs_ml_prep"]).all(), "coverage disagrees with sweep-library"

curves = {f"{c} {m}": (avail.filter((pl.col("split") == "test") & (pl.col("catalog") == c) & (pl.col("eligibility") == m) & pl.col("k").is_in(DEPTHS)))
          for c, m in [("library", "live"), ("library", "oracle")]}
plots.lines({n: (d["k"].to_list(), d["availability"].to_list()) for n, d in curves.items()},
            "OOS test pack availability against library depth k", "depth k (packs per key)", "positive transactions covered",
            PLOTS / "availability_vs_k.png", logx=True)
avail.filter(pl.col("k").is_in(DEPTHS)).pivot(on="k", index=["split", "catalog", "eligibility"], values="availability")

## 4. Route-recurrence ceiling

A pack can only cover RouteIds that it contains, and a frozen library only contains routes that were profitable in
training. The share of positives with *any* profitable RouteId seen among train positives therefore bounds what pack
retention can achieve.

In [ ]:
pos = tx.filter(pl.col("arb_target") == 1).with_columns(
    (pl.col("n_profitable_routes_seen_in_train") > 0).alias("route_seen"),
    (pl.col("min_cover_rank_oracle") < 64).fill_null(False).alias("covered_oracle_64"),
    (pl.col("min_cover_rank_live") < 64).fill_null(False).alias("covered_live_64"))
ceiling = pos.group_by("split").agg(
    pl.len().alias("positives"), pl.col("route_seen").sum().alias("route_seen_in_train"),
    pl.col("route_seen").mean().alias("ceiling"),
    pl.col("covered_oracle_64").sum().alias("covered_oracle_k64"), pl.col("covered_live_64").sum().alias("covered_live_k64"),
    (pl.col("covered_live_64") & pl.col("route_seen")).sum().alias("covered_live_given_seen"),
).sort("split")
RESULTS["route_recurrence"] = ceiling.to_dicts()
ceiling

## 5. Pack ranking: static vs learned

Each pair row is `transaction features + pack features`, and the target is whether the pack covers a profitable
RouteId of that transaction. The ranker is trained on the live pairs of POSITIVE train transactions (depth 64) and
early-stopped on validation. At weekly scale the full pair table (every partition, every label) never meets the
features at once: training rows are built from POSITIVE train/val pairs only, and val/test pairs are scored one
partition at a time, keeping only their scores.

Its output, P(pack covers | arb, tx, pack), is the pack term of the joint score.

Scorers compared, all deterministic, with ties broken by catalog rank and then pack id:

| scorer | rule |
|---|---|
| `library_static` | library rank |
| `ml` | learned |
| `ml_static_only` | learned, without the library's train_meta features (ablation) |
| `random` | seeded random |

**Keep rule:** the learned ranker is kept only if its validation top-1 beats `library_static` at depth 64.

In [ ]:
TX_COLS = ["master_tx_id", "split", *prep.tx_features]
PACK_COLS = ["pack_id", "builder_rank", "pack_layer", *prep.pack_features]
txf, packf = tx.select(TX_COLS), packs.select(PACK_COLS)
def with_features(p):
    out = p.join(txf, on="master_tx_id", how="inner").join(packf, on="pack_id", how="inner")
    if out.height != p.height:
        raise ValueError("pairs lost rows joining tx features or packs")
    return out

train_rows = []
for _, df in week.iter_pairs(W, (bounds["train"][0], bounds["val"][1])):
    df = df.filter((pl.col("arb_label") == "POSITIVE") & pl.col("eligible_live"))
    if RANKER_MAX_TXS:   # deterministic subsample of transactions, not rows: a transaction keeps all its packs
        df = df.filter((pl.col("master_tx_id") * 0x9E3779B97F4A7C15 % 1_000_003) < 1_000_003 * RANKER_MAX_TXS // max(1, int((tx["arb_target"] == 1).sum())))
    train_rows.append(with_features(df))
train_rows = pl.concat(train_rows)

def fit_ranker(cols):
    data.check_feature_columns(prep, cols)
    enc = encoding.FeatureEncoder(cols, prep.kinds).fit(train_rows.filter(pl.col("split") == "train"))
    tr, va = (train_rows.filter(pl.col("split") == s) for s in ("train", "val"))
    m = models.train_binary(enc.transform(tr), tr["pack_target"].to_numpy(), enc.transform(va), va["pack_target"].to_numpy(),
                            enc, RANKER_PARAMS)
    return m, enc
ranker, rk_enc = fit_ranker(prep.tx_features + prep.pack_features)
ranker_s, rk_enc_s = fit_ranker(prep.tx_features + STATIC_PACK)

# Val/test pairs, scored in 1M-row slices (a partition's encoded pairs alone are ~20 GB); only the columns
# the evaluation reads are kept.
KEEP = ["master_tx_id", "split", "pack_id", "pack_rank", "arb_label", "pack_target", "eligible_live"]
scored = []
for _, df in week.iter_pairs(W, (bounds["val"][0], bounds["test"][1])):
    for c in df.iter_slices(1_000_000):
        f = with_features(c)
        scored.append(f.select(KEEP).with_columns(
            pl.Series("ml", ranker.predict(rk_enc.transform(f), num_iteration=ranker.best_iteration)),
            pl.Series("ml_static_only", ranker_s.predict(rk_enc_s.transform(f), num_iteration=ranker_s.best_iteration))))
rf = pl.concat(scored)
del scored
rf = rf.with_columns((-pl.col("pack_rank").cast(pl.Float64)).alias("library_static"),
                     pl.Series("random", np.random.default_rng(7).random(rf.height)))
print("ranker rows: train", train_rows.filter(pl.col("split") == "train").height, "val", train_rows.filter(pl.col("split") == "val").height,
      "| scored val/test pairs", rf.height, "| best iterations", ranker.best_iteration, ranker_s.best_iteration)

In [ ]:
rows = []
for s in ("val", "test"):
    d = rf.filter(pl.col("split") == s)
    for depth in DEPTHS:
        for scorer in ("library_static", "ml", "ml_static_only", "random"):
            rows.append({"split": s, **policy.conditional_topk(d, scorer, depth, LIVE_TOP)})
ranking = pl.DataFrame(rows)
RESULTS["ranking"] = rows
v64 = {r["scorer"]: r["top1"] for r in rows if r["split"] == "val" and r["depth"] == 64}
KEEP_ML = (v64.get("ml") or 0) > (v64.get("library_static") or 0)
RESULTS["keep_ml_ranker"] = {"val_top1_depth64": v64, "keep": KEEP_ML}
display(Markdown(f"**Learned ranker kept: {KEEP_ML}.** Validation top-1 at depth 64: {v64}"))
ranking.select("split", "depth", "scorer", "coverable", "mean_eligible", "top1", "top2", "top3", "mrr", "top1_ci").sort("split", "depth", "scorer")

In [ ]:
RESULTS["ranker_importance"] = models.importance(ranker)[:40]
plots.importance(RESULTS["ranker_importance"], "Pack ranker feature importance", PLOTS / "ranker_importance.png")
t = ranking.filter(pl.col("split") == "test")
plots.lines({sc: (t.filter(pl.col("scorer") == sc)["depth"].to_list(), t.filter(pl.col("scorer") == sc)["top1"].to_list())
             for sc in ("library_static", "ml") if t.filter(pl.col("scorer") == sc).height},
            "OOS top-1 pack success given a covering pack exists", "depth k", "top-1 success", PLOTS / "ranking_top1_vs_k.png", logx=True);

## 6. Decision policies at matched send budgets

All policies use the same relevance gate and arb classifier. They differ in how they pick sends and packs:

| policy | decision | pack order |
|---|---|---|
| `library_two_stage` | arb ≥ t | library rank |
| `ml_two_stage` | arb ≥ t | learned ranker |
| `joint` | max over packs of P(arb) · P(pack) ≥ t | learned ranker |

For each send budget (sends per chain hour) and policy, the threshold is fitted on **validation**, then applied
unchanged to test. Depth k and live top-L are reported over a grid. The validation-best combination per budget is
the operating point.

In [ ]:
WEIGHT = policy.sample_weight(tx, PPM)
cols_tx = ["master_tx_id", "arb_target", "arb_label", "encode_status", GATE, "arb_score", "pair_sample", "min_cover_rank_live",
           "n_profitable_routes_seen_in_train", "split"]
tx_eval = tx.filter(pl.col("split") != "train").select(cols_tx)

POLICIES = {   # name -> (rule, pair frame, pack score, joint prob column)
    "library_two_stage": ("two_stage", rf, "library_static", None),
    "ml_two_stage": ("two_stage", rf, "ml", None),
    "joint": ("joint", rf, "ml", "ml"),
}

def frames(name, split, depth):
    """Per-transaction frames for every live top-L at one depth (one sort per depth)."""
    rule, pr, score, joint = POLICIES[name]
    t = tx_eval.filter(pl.col("split") == split)
    r = policy.ranked(pr.filter(pl.col("split") == split), score, depth)
    return {L: policy.per_tx(t, None, score, depth, L, joint, ranked_pairs=r) for L in LIVE_TOP}

rows = []
for name, (rule, *_rest) in POLICIES.items():
    for depth in DEPTHS:
        fv, ft = frames(name, "val", depth), frames(name, "test", depth)
        for L in LIVE_TOP:
            dv, dt = fv[L], ft[L]
            sv = policy.decision_score(dv, rule).to_numpy()
            wv = dv.select(WEIGHT).to_series().to_numpy()
            for budget in BUDGETS:
                t = policy.threshold_for_budget(sv, wv, budget * HOURS["val"])
                v = policy.evaluate(dv, rule, t, HOURS["val"], WEIGHT)
                o = policy.evaluate(dt, rule, t, HOURS["test"], WEIGHT)
                rows.append({"policy": name, "depth": depth, "L": L, "budget_per_hour": budget, "threshold": t,
                             "val_sends_per_hour": v["sends_per_chain_hour"], "val_end_to_end_recall": v["end_to_end_recall"],
                             "val_succeeded": v["succeeded"],
                             **{f"test_{k}": o[k] for k in ("sends_per_chain_hour", "precision_arb", "precision_capture",
                                                            "positive_recall", "availability", "topL_success_given_sent_and_available",
                                                            "end_to_end_recall", "end_to_end_recall_ci", "succeeded", "unknown_share_of_sends")},
                             **{f"test_missed_{k}": n for k, n in o["missed"].items()}})
grid = pl.DataFrame(rows)
RESULTS["policy_grid"] = rows
# Operating point per (policy, budget): the depth / L with the most validation successes (ties: smaller L, then smaller depth).
best = (grid.sort(["policy", "budget_per_hour", "val_succeeded", "L", "depth"], descending=[False, False, True, False, False])
            .group_by(["policy", "budget_per_hour"], maintain_order=True).first())
RESULTS["operating_points"] = best.to_dicts()
best.select("policy", "budget_per_hour", "depth", "L", "threshold", "val_succeeded", "test_sends_per_chain_hour",
            "test_precision_arb", "test_precision_capture", "test_positive_recall", "test_availability",
            "test_topL_success_given_sent_and_available", "test_succeeded", "test_end_to_end_recall", "test_end_to_end_recall_ci",
            "test_unknown_share_of_sends").sort("budget_per_hour", "policy")

In [ ]:
plots.lines({p: (best.filter(pl.col("policy") == p)["test_sends_per_chain_hour"].to_list(),
                 best.filter(pl.col("policy") == p)["test_end_to_end_recall"].to_list()) for p in POLICIES},
            "OOS end-to-end recall at validation-fitted send budgets", "test sends per chain hour", "end-to-end recall (all positives)",
            PLOTS / "recall_vs_sends.png", logx=True)
plots.lines({p: (best.filter(pl.col("policy") == p)["test_sends_per_chain_hour"].to_list(),
                 best.filter(pl.col("policy") == p)["test_precision_capture"].fill_null(0).to_list()) for p in POLICIES},
            "OOS capture precision of labelled sends", "test sends per chain hour", "labelled sends that capture an arb",
            PLOTS / "precision_vs_sends.png", logx=True);

## 7. Failure decomposition and conditional results

Every OOS positive transaction, at each policy's operating point, is assigned one outcome in pipeline order:
encoder cannot parse it (zero with framed-tx-v2; non-zero only for datasets built with framed-tx-v1) → relevance-gate miss → classifier/threshold miss
→ no covering candidate pack → pack-ranking miss → success.

In [ ]:
FOCUS_BUDGET = 1000
fb = best.filter(pl.col("budget_per_hour") == FOCUS_BUDGET)
bars = [{"label": f"{r['policy']} (k={r['depth']}, top-{r['L']})", "succeeded": r["test_succeeded"],
         "encoder_unsupported": r["test_missed_encoder_unsupported"],
         "relevance_gate": r["test_missed_relevance_gate"], "classifier": r["test_missed_threshold"],
         "no_candidate_pack": r["test_missed_no_covering_pack"], "ranker": r["test_missed_ranking"]} for r in fb.iter_rows(named=True)]
plots.miss_breakdown(bars, f"OOS positives by outcome at {FOCUS_BUDGET} sends per chain hour", PLOTS / "failure_decomposition.png")
cond = []
for r in fb.iter_rows(named=True):
    rule = POLICIES[r["policy"]][0]
    d = frames(r["policy"], "test", r["depth"])[r["L"]]
    for row in policy.conditional_success(d, rule, r["threshold"]).iter_rows(named=True):
        cond.append({"policy": r["policy"], **row})
cond = pl.DataFrame(cond)
RESULTS["conditional"] = cond.to_dicts()
cond.pivot(on="policy", index="subset", values="rate")

In [ ]:
h = {r["policy"]: r for r in fb.iter_rows(named=True)}
fmt = lambda x: "n/a" if x is None else f"{x:.2f}"
lines = [f"**At {FOCUS_BUDGET} sends per chain hour (thresholds fitted on validation), OOS test:**", ""]
for p, r in h.items():
    lines.append(f"- `{p}` (k={r['depth']}, top-{r['L']}): {r['test_succeeded']}/{r['test_succeeded'] + sum(r[c] for c in r if c.startswith('test_missed_'))} positives captured "
                 f"(end-to-end recall {fmt(r['test_end_to_end_recall'])}, CI {r['test_end_to_end_recall_ci']}), "
                 f"capture precision {fmt(r['test_precision_capture'])}, {r['test_sends_per_chain_hour']:.0f} sends/h.")
lines += ["", f"Learned ranker kept: **{KEEP_ML}**. Route-recurrence ceiling on test: "
          f"{ceiling.filter(pl.col('split') == 'test')['ceiling'][0]:.2f} of positives have any profitable RouteId seen in training."]
display(Markdown("\n".join(lines)))
RESULTS["headline"] = lines

## 8. Artifacts

In [ ]:
import shutil
sha = models.sha256_file
common = {"prepared_manifests_sha256": S["manifest_sha256"], "split_boundaries": bounds, "partitions": S["partition_indexes"],
          "feature_schema": prep.feature_manifest["feature_schema"], "pack_feature_schema": prep.feature_manifest["pack_feature_schema"],
          "pack_catalog": {k: pw[k] for k in ("catalog_schema", "count", "train_master_tx_id_min", "train_master_tx_id_max")},
          "relevance": {k: M["relevance"][k] for k in ("schema", "universe_digest", "rule")}}
arb_meta = models.save_model(arb, arb_enc, RUN_DIR / "models", "arb_classifier", ARB_PARAMS,
                             {**common, "target": "POSITIVE=1, CERTIFIED_NEGATIVE=0, UNKNOWN excluded", "applies_after_gate": GATE})
rk_meta = models.save_model(ranker, rk_enc, RUN_DIR / "models", "pack_ranker", RANKER_PARAMS,
                            {**common, "target": "pack covers a profitable RouteId", "training_rows": "live pairs of POSITIVE train transactions",
                             "kept": KEEP_ML})
ops = [{k: r[k] for k in ("policy", "budget_per_hour", "depth", "L", "threshold")} for r in best.iter_rows(named=True)]
(RUN_DIR / "operating_points.json").write_text(json.dumps(ops, indent=1))
shutil.copy(prep.root / "feature_manifest.json", RUN_DIR / "feature_manifest.json")
(RUN_DIR / "split_boundaries.json").write_text(json.dumps({"slots": bounds, "master_ids": ids, "library_window": pw["train_master_tx_id_max"]}, indent=1))
preds = RUN_DIR / "predictions"; preds.mkdir(exist_ok=True)
tx.filter(pl.col("split") != "train").select(cols_tx).write_parquet(preds / "arb_predictions.parquet")
rf.filter((pl.col("split") != "train") & pl.col("eligible_live")).select(
    "master_tx_id", "split", "pack_id", "pack_rank", "arb_label", "pack_target", "ml", "ml_static_only").write_parquet(preds / "pack_predictions.parquet")
(RUN_DIR / "metrics.json").write_text(json.dumps(RESULTS, indent=1, default=str))
artifacts = sorted(p for p in RUN_DIR.rglob("*") if p.is_file() and p.name != "experiment_manifest.json")
manifest = {
    "experiment": EXPERIMENT,
    "protocol": "library built from train master ids only and frozen; thresholds, depth and top-L from validation; test evaluated once",
    "prepared_dataset": {"path": str(WEEK), "partitions": S["partition_indexes"], "dataset_ids": S["dataset_ids"],
                         "manifests_sha256": S["manifest_sha256"], "library_windows": W.windows, "schemas": M["schemas"],
                         "packs": {k: v for k, v in pw.items() if k != "library"}, "library_manifest": pw.get("library"),
                         "raw_source": M["raw_source"]["kind"],
                         "provenance_files": {k: v["sha256"] for k, v in M["provenance"].items()}},
    "models": {"arb_classifier": arb_meta, "pack_ranker": rk_meta},
    "environment": models.environment(),
    "artifacts": [{"path": str(p.relative_to(RUN_DIR)).replace(os.sep, "/"), "sha256": sha(p)} for p in artifacts],
}
(RUN_DIR / "experiment_manifest.json").write_text(json.dumps(manifest, indent=1, default=str))
b, e, meta = models.load_model(RUN_DIR / "models", "pack_ranker")
assert meta["feature_order"] == prep.tx_features + prep.pack_features
print(RUN_DIR); print("\n".join(a["path"] for a in manifest["artifacts"]))

### Caveats
- **One week, one regime.** Test is the last ~15% of the same week. Whether a model trained on one week holds on
  the next is only measured once a second week is labelled.
- **Partitions labelled so far.** The run covers the labelled partitions in `windows.json`; unlabelled partitions
  of the week are absent, not negative.
- **Joint score.** P(pack) comes from a ranker trained on positive transactions only. The product is therefore
  P(arb) · P(cover | arb), not a separately calibrated joint probability.
- **Live volume.** It is estimated from the deterministic UNKNOWN pair sample (`unknown_pair_ppm`) and weighted by
  1/rate. Failed transactions and transactions outside the master set are not in the population.
- **Library metadata.** The train_meta pack features come from the library's training window. They are valid only
  because the notebook asserts that window ends before validation.